# NanoQA v4 — Definitive Fixed Training Notebook
## What was wrong in every previous run and exactly how this fixes it

### Bug 1 (Original v3 notebook): DataCollatorForLanguageModeling overwrote labels
The original notebook carefully built `-100` masks for question tokens, then passed the dataset to
`DataCollatorForLanguageModeling` which silently replaced all those labels with its own random masking.
**Fix:** Custom training loop that never touches labels after construction.

### Bug 2 (All runs): Data augmentation before train/val split = data leakage
Augmenting all pairs first, then splitting means augmented variants of val/test questions appear in training.
**Fix:** Split on original base pairs first. Augment only the training portion.

### Bug 3 (Runs 1 & 2): Answer prefix included trailing space → space token loop
`"\nAnswer: "` tokenizes as `[198, 33706, 25, 220]`. Token 220 (space) was masked. At inference the model 
saw token 220 and had never learned what follows it → infinite space loop.
**Fix:** `ANSWER_PREFIX = "\nAnswer:"` — no trailing space. The space is BPE-merged into the first answer token 
(e.g. `" William"` is token 3977, not space + William). Training and inference use identical prefix.

### Bug 4 (Run 3): Removed distillation → catastrophic overfitting in 2 epochs
Without GPT-2's soft distributions as regularizer, train loss hit 0.0000 while val stayed at 0.17.
**Fix:** Keep distillation but reduce weight (α=0.7 focal, 0.3 KL). Also lower LR to 1e-4.

### Verification: The notebook asserts the correct prefix tokens before training even starts.
If the assert fails, you see the problem immediately — not after 5 hours.

---
**Kaggle setup:**
1. Runtime → GPU T4 or T4×2
2. Internet ON
3. Attach dataset `nanoqa-handcrafted` containing `handcrafted_qa.json` and `new_handcrafted_qa.json`
4. Run All Cells


In [1]:
# ── CELL 1: Install & GPU check ───────────────────────────────────────────────
# This installs everything needed. Run once, then proceed.

import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "datasets", "accelerate"], check=True)

import torch, math, json, os, random, time
from pathlib import Path
from torch import nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — training will be very slow")

# Reproducibility — same seeds = same results every time
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seeds set:", SEED)


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Seeds set: 42


In [2]:
# ── CELL 2: All hyperparameters in one place ───────────────────────────────────
# Change values HERE only. Every cell below reads from these variables.

# Paths — update if your Kaggle dataset has a different name
HC_PATH1 = "/kaggle/input/datasets/venisatellis/nanoqa-handcrafted-qa/handcrafted_qa.json"
HC_PATH2 = "/kaggle/input/datasets/venisatellis/new-handcrafted-qa-json/new_handcrafted_qa.json"
OUTPUT_DIR = "/kaggle/working/nanoqa_v4"

# ── THE CRITICAL FIX ──────────────────────────────────────────────────────────
# "\nAnswer:" with NO trailing space.
# Why: GPT-2 BPE encodes " William" as a single token (3977), not space+William.
# Training labels start at that " William" token. Inference prompt must end at ":"
# so the model generates " William" naturally as its first token.
# With "\nAnswer: " (space), the model sees token 220 and was never trained on
# what follows a standalone space → infinite space loop.
ANSWER_PREFIX = "\nAnswer:"

# Model architecture (matches paper Table I exactly)
VOCAB_SIZE  = 50257   # GPT-2 BPE vocabulary
HIDDEN_SIZE = 768     # d_model
NUM_LAYERS  = 12      # transformer depth
NUM_HEADS   = 12      # attention heads (head_dim = 64)
FF_SIZE     = 3072    # 4 × hidden_size
MAX_SEQ_LEN = 128     # max tokens per sample
DROPOUT     = 0.1

# Training
LEARNING_RATE  = 1e-4   # lower than previous runs to prevent overfitting
BATCH_SIZE     = 64   # per-device; effective batch = BATCH_SIZE × GRAD_ACCUM
GRAD_ACCUM     = 2     # effective batch = 128
NUM_EPOCHS     = 5
WARMUP_STEPS   = 500
WEIGHT_DECAY   = 0.01
GRAD_CLIP      = 1.0
EVAL_EVERY     = 500    # steps between validation
ES_PATIENCE    = 4      # early stopping patience (in eval intervals)
USE_FP16       = torch.cuda.is_available()

# Loss: focal_weight × focal + (1-focal_weight) × KL_distillation
FOCAL_GAMMA    = 2.0
FOCAL_WEIGHT   = 0.92    # 0.7 focal + 0.3 distillation
DISTILL_TEMP   = 3.0    # temperature for soft targets (higher = softer = more info)

# Data augmentation (applied to TRAIN split only — no leakage)
AUGMENT_TRAIN  = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Config loaded. Output directory:", OUTPUT_DIR)
print(f"ANSWER_PREFIX = {repr(ANSWER_PREFIX)}")


Config loaded. Output directory: /kaggle/working/nanoqa_v4
ANSWER_PREFIX = '\nAnswer:'


In [3]:
# ── CELL 3: NanoQA v4 Architecture ────────────────────────────────────────────
# Causal decoder-only transformer, built entirely from scratch in PyTorch.
# No HuggingFace PreTrainedModel base class — avoids safetensors memory-sharing bug.
# Architecture exactly matches paper Table I: 12 layers, 768 hidden, 12 heads, pre-norm.

class NanoQAConfig:
    """All architecture hyperparameters in one place. Read from Cell 2 variables."""
    def __init__(self):
        self.vocab_size  = VOCAB_SIZE
        self.hidden_size = HIDDEN_SIZE
        self.num_layers  = NUM_LAYERS
        self.num_heads   = NUM_HEADS
        self.ff_size     = FF_SIZE
        self.max_seq_len = MAX_SEQ_LEN
        self.dropout     = DROPOUT


class NanoQAAttention(nn.Module):
    """
    Multi-head causal self-attention.
    - Fused QKV projection: one matrix for Q, K, V — efficient on GPU
    - Causal mask: prevents token i from attending to tokens > i (autoregressive)
    - head_dim = hidden_size / num_heads = 768 / 12 = 64
    """
    def __init__(self, cfg):
        super().__init__()
        assert cfg.hidden_size % cfg.num_heads == 0, "hidden_size must be divisible by num_heads"
        self.num_heads = cfg.num_heads
        self.head_dim  = cfg.hidden_size // cfg.num_heads
        self.hidden    = cfg.hidden_size
        # Single matrix for Q, K, V — 3× the hidden size
        self.qkv  = nn.Linear(cfg.hidden_size, 3 * cfg.hidden_size, bias=False)
        self.proj = nn.Linear(cfg.hidden_size, cfg.hidden_size, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
        # Lower-triangular causal mask: position i can only see positions ≤ i
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len))
            .view(1, 1, cfg.max_seq_len, cfg.max_seq_len)
        )

    def forward(self, x):
        B, T, C = x.shape
        # Split fused QKV into three equal parts
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Reshape to (Batch, Heads, Time, HeadDim) for batched dot-product
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        # Scaled dot-product attention with causal masking
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = self.drop(F.softmax(att, dim=-1))
        # Merge heads back to (B, T, C)
        return self.proj((att @ v).transpose(1, 2).contiguous().view(B, T, C))


class NanoQAFFN(nn.Module):
    """
    Feed-forward network: two linear layers with GELU activation.
    Expands to 4× hidden (3072) then contracts back.
    GELU is smoother than ReLU — helps with gradient flow from random init.
    """
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.ff_size),
            nn.GELU(),
            nn.Linear(cfg.ff_size, cfg.hidden_size),
            nn.Dropout(cfg.dropout),
        )
    def forward(self, x): return self.net(x)


class NanoQABlock(nn.Module):
    """
    Single transformer block: pre-norm → attention → residual → pre-norm → FFN → residual.
    Pre-norm (LayerNorm BEFORE attention) stabilises gradient flow from random init —
    critical since we have no pre-trained weights to start from.
    """
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)
        self.att = NanoQAAttention(cfg)
        self.ffn = NanoQAFFN(cfg)

    def forward(self, x):
        x = x + self.att(self.ln1(x))  # attention with pre-norm + residual
        x = x + self.ffn(self.ln2(x))  # FFN with pre-norm + residual
        return x


class NanoQA(nn.Module):
    """
    Full NanoQA v4 causal language model.
    
    forward() returns logits (B, T, vocab_size) — used during training.
    generate() returns (token_ids_list, confidence_scores) — used during inference.
    
    Confidence is computed as raw softmax probability of each chosen token
    BEFORE temperature scaling, exactly as described in paper Eq. (2)-(3).
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([NanoQABlock(cfg) for _ in range(cfg.num_layers)])
        self.ln_f    = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        # Weight tying: input embedding and output projection share weights
        # Reduces params and improves generalization (standard practice)
        self.lm_head.weight = self.tok_emb.weight
        self._init_weights()
        total = sum(p.numel() for p in self.parameters())
        print(f"NanoQA v4: {total:,} parameters ({total/1e6:.1f}M)")

    def _init_weights(self):
        """
        GPT-style weight init: N(0, 0.02) for linear/embedding, ones/zeros for LayerNorm.
        Residual projections scaled by 1/sqrt(2*num_layers) to control variance growth.
        """
        for name, p in self.named_parameters():
            if 'proj' in name and 'weight' in name:  # residual projection
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * self.cfg.num_layers))
            elif 'weight' in name and p.dim() >= 2:
                nn.init.normal_(p, 0.0, 0.02)
            elif 'bias' in name:
                nn.init.zeros_(p)
            elif 'ln' in name and 'weight' in name:  # LayerNorm scale
                nn.init.ones_(p)

    def forward(self, x):
        """x: (B, T) token ids → logits: (B, T, vocab_size)"""
        B, T = x.shape
        assert T <= self.cfg.max_seq_len, f"Sequence too long: {T} > {self.cfg.max_seq_len}"
        pos    = torch.arange(T, device=x.device)
        hidden = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        for block in self.blocks:
            hidden = block(hidden)
        return self.lm_head(self.ln_f(hidden))  # (B, T, vocab)

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens=30, temperature=0.3,
                 top_k=10, eos_token_id=None):
        """
        Autoregressively generate tokens one at a time.
        Returns: (generated_ids: list[int], confidences: list[float])
        
        Confidence = raw softmax prob of chosen token BEFORE temperature.
        This is what the router uses for the routing decision.
        """
        self.eval()
        ids  = prompt_ids.clone()
        gen  = []
        conf = []

        for _ in range(max_new_tokens):
            # Only feed last max_seq_len tokens (sliding window)
            ctx = ids[:, -self.cfg.max_seq_len:]
            logits = self(ctx)               # (1, T, vocab)
            next_logits = logits[0, -1, :]   # logits for next token

            # ── Confidence: raw softmax BEFORE temperature ─────────────────
            # This is Eq. (2) from the paper: p_raw(t_i) = softmax(l_ti)
            raw_probs = F.softmax(next_logits, dim=-1)

            # ── Sampling: apply temperature then top-k ─────────────────────
            scaled = next_logits / max(temperature, 1e-8)
            if top_k > 0:
                topk_vals, _ = torch.topk(scaled, min(top_k, scaled.size(-1)))
                scaled[scaled < topk_vals[-1]] = float('-inf')
            probs    = F.softmax(scaled, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).item()

            if eos_token_id is not None and next_tok == eos_token_id:
                break

            conf.append(raw_probs[next_tok].item())
            gen.append(next_tok)
            ids = torch.cat([ids, torch.tensor([[next_tok]], device=ids.device)], dim=1)

        return gen, conf


# Build model
cfg   = NanoQAConfig()
model = NanoQA(cfg).to(DEVICE)


NanoQA v4: 123,714,816 parameters (123.7M)


In [4]:
# ── CELL 4: Tokenizer + Critical Prefix Verification ──────────────────────────
# This cell verifies the ANSWER_PREFIX tokens are correct BEFORE training starts.
# If this assert fails, you see it immediately — not after 5 hours.

from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# ── Verify the prefix tokenization ────────────────────────────────────────────
prefix_ids = tokenizer.encode(ANSWER_PREFIX, add_special_tokens=False)
print(f"ANSWER_PREFIX = {repr(ANSWER_PREFIX)}")
print(f"Prefix tokens : {prefix_ids}")
print(f"Decoded       : {[repr(tokenizer.decode([t])) for t in prefix_ids]}")

# Example: show what the FIRST answer token looks like
example = f"Question: Who wrote Romeo and Juliet?{ANSWER_PREFIX} William Shakespeare."
all_ids = tokenizer.encode(example, add_special_tokens=False)
ans_start = len(prefix_ids) + len(tokenizer.encode("Question: Who wrote Romeo and Juliet?", add_special_tokens=False))
print(f"\nExample full sequence tokens: {all_ids}")
print(f"Answer starts at position    : {ans_start}")
print(f"First answer token           : {all_ids[ans_start]} = {repr(tokenizer.decode([all_ids[ans_start]]))}")
print(f"  (should be ' William' with leading space — BPE merges space+word)")

# Critical assertion: last prefix token must be 25 (:)
# This ensures inference prompt ends at ":" and model generates " William" next
assert prefix_ids[-1] == 25, f"Expected last prefix token to be 25 (:), got {prefix_ids[-1]}"
print("\n✓ ANSWER_PREFIX assertion passed — prefix ends correctly at ':' (token 25)")
print("  Training and inference will use identical token boundaries.")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ANSWER_PREFIX = '\nAnswer:'
Prefix tokens : [198, 33706, 25]
Decoded       : ["'\\n'", "'Answer'", "':'"]

Example full sequence tokens: [24361, 25, 5338, 2630, 43989, 290, 38201, 30, 198, 33706, 25, 3977, 22197, 13]
Answer starts at position    : 11
First answer token           : 3977 = ' William'
  (should be ' William' with leading space — BPE merges space+word)

✓ ANSWER_PREFIX assertion passed — prefix ends correctly at ':' (token 25)
  Training and inference will use identical token boundaries.


In [5]:
# ── CELL 5: Load Handcrafted Datasets ─────────────────────────────────────────
# Loads your two handcrafted JSON files.
# DOES NOT augment yet — augmentation happens only AFTER the train/val/test split.

import json, os

def load_json_qa(path, name):
    if not os.path.exists(path):
        print(f"  WARNING: {name} not found at {path} — skipping")
        return []
    with open(path) as f:
        data = json.load(f)
    pairs = []
    for item in data:
        q = item.get("question", "").strip()
        a = item.get("answer", "").strip()
        if q and a:
            pairs.append((q, a))
    print(f"  {name}: {len(pairs):,} pairs")
    return pairs

print("Loading handcrafted datasets...")
hc1 = load_json_qa(HC_PATH1, "handcrafted_qa")
hc2 = load_json_qa(HC_PATH2, "new_handcrafted_qa")
handcrafted_base = hc1 + hc2

# Deduplicate by question
seen_q = set()
hc_deduped = []
for q, a in handcrafted_base:
    k = q.lower().strip()
    if k not in seen_q:
        seen_q.add(k)
        hc_deduped.append((q, a))

print(f"  Combined after dedup: {len(hc_deduped):,} base pairs")


Loading handcrafted datasets...
  handcrafted_qa: 32,221 pairs
  new_handcrafted_qa: 2,135 pairs
  Combined after dedup: 33,818 base pairs


In [6]:
# ── CELL 6: Load TriviaQA and SQuAD ───────────────────────────────────────────
# Two external datasets that add linguistic diversity — different question styles,
# different phrasing patterns, preventing the model from only learning your templates.
# 
# TriviaQA rc.nocontext: closed-book trivia — model answers from memory, no passage
# SQuAD: Wikipedia QA — filtered to 1-8 word answers to match NanoQA's output style

from datasets import load_dataset

def load_trivia(n=75000):
    print("Loading TriviaQA...")
    ds = load_dataset("trivia_qa", "rc.nocontext", split="train", trust_remote_code=False)
    pairs = []
    for item in ds:
        q = item["question"].strip()
        a = item["answer"]["value"].strip()
        if q and a and len(a.split()) <= 8:  # short answers only
            pairs.append((q, a))
        if len(pairs) >= n:
            break
    print(f"  TriviaQA: {len(pairs):,} pairs")
    return pairs

def load_squad(n=35000):
    print("Loading SQuAD...")
    ds = load_dataset("squad", split="train")
    pairs = []
    seen = set()
    for item in ds:
        q = item["question"].strip()
        a = item["answers"]["text"][0].strip() if item["answers"]["text"] else ""
        if not q or not a:
            continue
        word_count = len(a.split())
        if word_count < 1 or word_count > 8:  # filter to short answers
            continue
        k = q.lower()[:60]
        if k in seen:
            continue
        seen.add(k)
        pairs.append((q, a))
        if len(pairs) >= n:
            break
    print(f"  SQuAD: {len(pairs):,} pairs")
    return pairs

trivia_base = load_trivia(75000)
squad_base  = load_squad(35000)

all_base = hc_deduped + trivia_base + squad_base
print(f"\nTotal base pairs: {len(all_base):,}")


Loading TriviaQA...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc.nocontext/train-00000-of-00001.parque(…):   0%|          | 0.00/55.4M [00:00<?, ?B/s]

rc.nocontext/validation-00000-of-00001.p(…):   0%|          | 0.00/7.34M [00:00<?, ?B/s]

...

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

  SQuAD: 35,000 pairs

Total base pairs: 143,818


In [7]:
# ── CELL 7: Split FIRST → Augment TRAIN ONLY (No Data Leakage) ────────────────
#
# WHY ORDER MATTERS:
# Wrong order (original bug): augment all → split → model sees augmented variants
#   of val/test questions during training → inflated metrics
# Correct order: split originals → augment only train → val/test are clean originals
#
# The split is done on INDICES, not on text, so augmented variants
# can never appear in val or test.

import random
random.seed(SEED)

# Shuffle then split: 92% train, 4% val, 4% test
random.shuffle(all_base)
n = len(all_base)
n_val  = max(500, int(n * 0.04))
n_test = max(500, int(n * 0.04))
n_train = n - n_val - n_test

train_base = all_base[:n_train]
val_base   = all_base[n_train:n_train + n_val]
test_base  = all_base[n_train + n_val:]

print(f"Base split:")
print(f"  Train : {len(train_base):,} original pairs")
print(f"  Val   : {len(val_base):,} original pairs (clean, no augmentation)")
print(f"  Test  : {len(test_base):,} original pairs (clean, never touched until final eval)")


def augment_pair(q, a):
    """
    Generate 8 surface variants of one QA pair.
    All variants have the SAME answer — only the question phrasing changes.
    This teaches the model to handle different question styles.
    """
    variants = [(q, a)]                                    # 1. original
    ql = q.strip().rstrip('?').lower()
    variants.append((ql + '?', a))                         # 2. lowercase
    variants.append((ql.upper() + '?', a))                 # 3. uppercase
    qc = ql.replace('what is', "what's").replace('who is', "who's")
    if qc != ql:
        variants.append((qc + '?', a))                     # 4. contracted
    qs = ql
    for pfx in ['what is the ', 'what is a ', 'who is the ', 'who was the ',
                'when did ', 'when was ', 'how many ', 'what are the ']:
        if qs.startswith(pfx):
            qs = qs[len(pfx):]
            break
    if qs != ql:
        variants.append((qs + '?', a))                     # 5. short-form
    variants.append((f'tell me {ql}', a))                  # 6. tell me
    variants.append((f'can you tell me {ql}?', a))         # 7. can you tell me
    if any(ql.startswith(w) for w in ['who', 'what', 'when', 'where']):
        variants.append((f'do you know {ql}?', a))         # 8. do you know
    if len(ql) > 8:                                        # 9. typo simulation
        pos = random.randint(2, len(ql) - 3)
        typo = ql[:pos] + ql[pos+1:] + '?'
        variants.append((typo, a))
    return variants


def pairs_to_texts(pairs, augment=False):
    """Convert (question, answer) pairs to formatted training strings."""
    texts = []
    for q, a in pairs:
        variants = augment_pair(q, a) if augment else [(q, a)]
        for vq, va in variants:
            # Format: "Question: {q}\nAnswer: {a}" — this is the training text
            # Note the space after ANSWER_PREFIX is part of the answer,
            # not the prefix: " William" is one BPE token
            texts.append(f"Question: {vq}{ANSWER_PREFIX} {va}")
    return texts

if AUGMENT_TRAIN:
    train_texts = pairs_to_texts(train_base, augment=True)
    print(f"\nAfter augmentation:")
    print(f"  Train : {len(train_texts):,} samples (original × ~9 variants)")
else:
    train_texts = pairs_to_texts(train_base, augment=False)
    print(f"  Train : {len(train_texts):,} samples (no augmentation)")

val_texts  = pairs_to_texts(val_base,  augment=False)
test_texts = pairs_to_texts(test_base, augment=False)

print(f"  Val   : {len(val_texts):,} samples")
print(f"  Test  : {len(test_texts):,} samples")
print(f"\nExample training text:")
print(f"  {repr(train_texts[0][:100])}")


Base split:
  Train : 132,314 original pairs
  Val   : 5,752 original pairs (clean, no augmentation)
  Test  : 5,752 original pairs (clean, never touched until final eval)

After augmentation:
  Train : 883,312 samples (original × ~9 variants)
  Val   : 5,752 samples
  Test  : 5,752 samples

Example training text:
  'Question: Descending letters tend to have a part which falls where relative to the average height of'


In [8]:
# ── CELL 8: QA Dataset with Answer-Only Loss Masking ──────────────────────────
#
# Answer-only masking: question tokens get label = -100 (PyTorch ignores these)
# Only answer tokens contribute to the gradient.
# Without this, the model wastes gradient budget learning to predict question words.
#
# HOW IT WORKS:
#   Full text: "Question: Who wrote Romeo?\nAnswer: William Shakespeare."
#   Labels:    [-100, -100, ..., -100,  William_tok, Shakespeare_tok, ._tok, eos_tok]
#              ←── question tokens ───→  ←────── answer tokens ──────────────────────→

import torch
from torch.utils.data import Dataset, DataLoader

# Get the token IDs of the answer prefix — these will be searched for in each sequence
PREFIX_IDS = tokenizer.encode(ANSWER_PREFIX, add_special_tokens=False)
print(f"Prefix IDs to search for: {PREFIX_IDS}")
print(f"Decoded: {[repr(tokenizer.decode([t])) for t in PREFIX_IDS]}")

# Double-check: last token must be 25 (:)
assert PREFIX_IDS[-1] == 25, f"Bad prefix: {PREFIX_IDS}"
print("✓ Prefix verified in dataset class")


def find_answer_start(token_ids):
    """
    Find the position immediately after the answer prefix in a token sequence.
    Returns the index of the FIRST answer token (the one after ANSWER_PREFIX).
    If prefix not found, returns len(token_ids) (all tokens masked).
    """
    n = len(PREFIX_IDS)
    for i in range(len(token_ids) - n + 1):
        if token_ids[i:i+n] == PREFIX_IDS:
            return i + n  # first answer token position
    return len(token_ids)  # prefix not found → mask everything


class QADataset(Dataset):
    """
    Tokenizes text with answer-only masking.
    Each item: input_ids (MAX_SEQ_LEN,), labels (MAX_SEQ_LEN,)
    Labels: -100 for question/prefix tokens, actual token ids for answer tokens.
    """
    def __init__(self, texts, max_len=MAX_SEQ_LEN):
        self.samples = []
        skipped = 0
        for text in texts:
            # Add EOS token to signal end of answer
            full = text + tokenizer.eos_token
            ids  = tokenizer.encode(full, add_special_tokens=False)
            if len(ids) > max_len:
                skipped += 1
                continue  # skip sequences that don't fit

            # Pad to max_len
            pad_len = max_len - len(ids)
            padded  = ids + [tokenizer.pad_token_id] * pad_len

            # Build labels: -100 everywhere, then fill answer tokens
            labels     = [-100] * max_len
            ans_start  = find_answer_start(ids)
            for j in range(ans_start, len(ids)):
                labels[j] = ids[j]

            self.samples.append({
                "input_ids": torch.tensor(padded, dtype=torch.long),
                "labels":    torch.tensor(labels,  dtype=torch.long),
            })
        print(f"  Dataset: {len(self.samples):,} samples ({skipped} skipped — too long)")

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


# Verify masking is working correctly on one example
def verify_masking():
    example = train_texts[0]
    print(f"Example text: {repr(example[:80])}")
    full_ids = tokenizer.encode(example + tokenizer.eos_token, add_special_tokens=False)
    ans_start = find_answer_start(full_ids)
    print(f"Token IDs: {full_ids}")
    print(f"Answer starts at position {ans_start}: token {full_ids[ans_start] if ans_start < len(full_ids) else 'N/A'}")
    if ans_start < len(full_ids):
        print(f"First answer token decoded: {repr(tokenizer.decode([full_ids[ans_start]]))}")
        print("✓ Masking verified — first answer token is correct")
    else:
        print("WARNING: answer prefix not found in example")

verify_masking()

print("\nBuilding datasets...")
train_ds = QADataset(train_texts)
val_ds   = QADataset(val_texts)
test_ds  = QADataset(test_texts)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2)

print(f"Train batches: {len(train_loader):,}")
print(f"Val   batches: {len(val_loader):,}")


Prefix IDs to search for: [198, 33706, 25]
Decoded: ["'\\n'", "'Answer'", "':'"]
✓ Prefix verified in dataset class
Example text: 'Question: Descending letters tend to have a part which falls where relative to t'
Token IDs: [24361, 25, 39373, 1571, 7475, 4327, 284, 423, 257, 636, 543, 8953, 810, 3585, 284, 262, 2811, 6001, 286, 257, 3850, 30, 198, 33706, 25, 2793, 621, 262, 2811, 50256]
Answer starts at position 25: token 2793
First answer token decoded: ' lower'
✓ Masking verified — first answer token is correct

Building datasets...
  Dataset: 883,166 samples (146 skipped — too long)
  Dataset: 5,752 samples (0 skipped — too long)
  Dataset: 5,752 samples (0 skipped — too long)
Train batches: 13,800
Val   batches: 90


In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 CHANGE — update these two lines only:
# ══════════════════════════════════════════════════════════════════════
# BATCH_SIZE = 64     # <-- was 16, change to 64
# GRAD_ACCUM = 2      # <-- was 8,  change to 2  (effective batch still 128)
#
# WHY: with BATCH_SIZE=16 and heavy answer-only masking, many batches have
# near-zero answer tokens → noisy/zero gradients → loss stays high.
# Larger batches = more answer tokens per gradient step = stable training.
# ══════════════════════════════════════════════════════════════════════


# ── CELL 9 REPLACEMENT ────────────────────────────────────────────────────────
from transformers import GPT2LMHeadModel
import torch, torch.nn.functional as F

def focal_loss(logits, labels, gamma=FOCAL_GAMMA):
    """
    Token-level focal loss — answer tokens only.
    logits: (B, T, V) | labels: (B, T) with -100 for question tokens
    """
    B, T, V = logits.shape
    lf = logits[:, :-1, :].contiguous().view(-1, V)   # (B*(T-1), V)
    la = labels[:, 1:].contiguous().view(-1)           # (B*(T-1),)
    mask = (la != -100)

    # If zero answer tokens in this batch, return zero WITH grad
    # (not a detached 0.0 — that kills the backward pass)
    if mask.sum() == 0:
        return (logits * 0).sum()

    lf_ans = lf[mask]        # (N_ans, V)
    la_ans = la[mask]        # (N_ans,)

    ce = F.cross_entropy(lf_ans, la_ans, reduction='none')  # (N_ans,)

    with torch.no_grad():
        pt = F.softmax(lf_ans, dim=-1).gather(1, la_ans.unsqueeze(1)).squeeze(1)
        w  = (1.0 - pt.clamp(1e-6, 1.0)) ** gamma

    return (w * ce).mean()


def combined_loss(student_logits, labels, input_ids):
    """
    FOCAL_WEIGHT × focal(answer tokens) + (1-FOCAL_WEIGHT) × KL(answer tokens)
    Both losses restricted to answer tokens — same token set, comparable scale.
    """
    fl = focal_loss(student_logits, labels)

    with torch.no_grad():
        t_logits = teacher(input_ids).logits

    B, T, V = student_logits.shape
    s_shift  = student_logits[:, :-1, :].contiguous()   # (B, T-1, V)
    t_shift  = t_logits[:, :-1, :].contiguous()
    la_shift = labels[:, 1:].contiguous()                # (B, T-1)
    ans_mask = (la_shift != -100).view(-1)               # (B*(T-1),)

    if ans_mask.sum() == 0:
        # No answer tokens — skip KL, return focal only (with grad)
        return FOCAL_WEIGHT * fl, fl.item(), 0.0

    s_flat = s_shift.view(-1, V)[ans_mask]   # (N_ans, V)
    t_flat = t_shift.view(-1, V)[ans_mask]   # (N_ans, V)

    s_log = F.log_softmax(s_flat / DISTILL_TEMP, dim=-1)
    t_sft = F.softmax(t_flat / DISTILL_TEMP, dim=-1)

    # sum then divide by N_ans manually — 'batchmean' would divide by N_ans
    # anyway, but being explicit avoids PyTorch version differences
    kl = (t_sft * (t_sft.log() - s_log)).sum() / ans_mask.sum().float()
    kl = kl * (DISTILL_TEMP ** 2)

    total = FOCAL_WEIGHT * fl + (1.0 - FOCAL_WEIGHT) * kl
    return total, fl.item(), kl.item()


print("Loading frozen GPT-2 teacher...")
teacher = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print(f"Teacher: {sum(p.numel() for p in teacher.parameters()):,} params — FROZEN")
print(f"Loss: {FOCAL_WEIGHT:.0%} focal + {1-FOCAL_WEIGHT:.0%} KL | "
      f"gamma={FOCAL_GAMMA} | T={DISTILL_TEMP}")
print("✓ Both losses answer-token-only | ✓ Zero-batch handled with grad")

# ── Sanity check: loss, grad flow, focal/KL ratio ─────────────────────────────
print("\nRunning sanity check...")
model.train()
dummy_ids    = torch.randint(100, 50000, (4, 128)).to(DEVICE)
dummy_labels = torch.full((4, 128), -100, dtype=torch.long).to(DEVICE)
# Put answer tokens at positions 50-60 (simulates a short answer)
dummy_labels[:, 50:60] = dummy_ids[:, 50:60]

dummy_logits = model(dummy_ids)
loss_val, fl_val, kl_val = combined_loss(dummy_logits, dummy_labels, dummy_ids)

# Check backward doesn't crash
loss_val.backward()
model.zero_grad()

print(f"  Total: {loss_val:.4f} | Focal: {fl_val:.4f} | KL: {kl_val:.4f}")
ratio = fl_val / (kl_val + 1e-8)
print(f"  Focal/KL ratio: {ratio:.2f}x  (healthy range: 0.3x – 5x)")
assert not torch.isnan(loss_val), "NaN loss"
assert loss_val.item() < 20.0,   f"Loss too high: {loss_val:.2f}"
print("✓ Sanity check passed — loss is finite, backward works")

Loading frozen GPT-2 teacher...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Teacher: 124,439,808 params — FROZEN
Loss: 92% focal + 8% KL | gamma=2.0 | T=3.0
✓ Both losses answer-token-only | ✓ Zero-batch handled with grad

Running sanity check...
  Total: 10.2820 | Focal: 11.0120 | KL: 1.8861
  Focal/KL ratio: 5.84x  (healthy range: 0.3x – 5x)
✓ Sanity check passed — loss is finite, backward works


In [10]:
# ── CELL 10: Custom Training Loop ─────────────────────────────────────────────
# Fixed: replaced LambdaLR (was returning ~0 due to step mismatch) with
# CosineAnnealingLR which works correctly with the gradient accumulation pattern.

import torch, math, os, time
from torch.cuda.amp import GradScaler, autocast

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    betas        = (0.9, 0.999),
)

total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS

def get_lr(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)

print(f"Steps per epoch : {len(train_loader) // GRAD_ACCUM:,}")
print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Total steps     : {total_steps:,}")
print(f"Peak LR         : {LEARNING_RATE:.1e}")
print(f"Min LR          : {LEARNING_RATE * 0.01:.1e}")
print(f"fp16            : {USE_FP16}")


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    n = 0
    for batch in loader:
        ids    = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_FP16):
            logits = model(ids)
            loss, _, _ = combined_loss(logits, labels, ids)
        total_loss += loss.item()
        n += 1
    model.train()
    avg = total_loss / max(n, 1)
    return avg, math.exp(min(avg, 20))


history    = []
best_val   = float('inf')
no_improve = 0
global_step = 0
best_ckpt  = os.path.join(OUTPUT_DIR, "nanoqa_v4_best.pt")
start_time = time.time()

print("\n" + "="*70)
print(f"Training NanoQA v4 | {len(train_ds):,} samples | {NUM_EPOCHS} epochs")
print("="*70)
model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    n_steps    = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast('cuda', enabled=USE_FP16):
            logits = model(ids)
            loss, fl_val, kl_val = combined_loss(logits, labels, ids)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step  += 1
            epoch_loss   += loss.item() * GRAD_ACCUM
            n_steps      += 1

            if global_step % EVAL_EVERY == 0:
                val_loss, val_ppl = evaluate(val_loader)
                elapsed  = (time.time() - start_time) / 60
                lr_now   = scheduler.get_last_lr()[0]
                print(f"Step {global_step:5d} | Ep {epoch}/{NUM_EPOCHS} | "
                      f"Train {epoch_loss/n_steps:.4f} | Val {val_loss:.4f} | "
                      f"PPL {val_ppl:.2f} | LR {lr_now:.2e} | {elapsed:.1f}min")

                history.append({
                    "step": global_step, "epoch": epoch,
                    "train": epoch_loss / n_steps, "val_loss": val_loss,
                    "perplexity": val_ppl, "lr": lr_now,
                })

                if val_loss < best_val:
                    best_val   = val_loss
                    no_improve = 0
                    torch.save({
                        "model_state_dict": model.state_dict(),
                        "val_loss":         best_val,
                        "epoch":            epoch,
                        "step":             global_step,
                        "config":           cfg.__dict__,
                        "answer_prefix":    ANSWER_PREFIX,
                    }, best_ckpt)
                    print(f"  ✓ Best saved (val={best_val:.4f})")
                else:
                    no_improve += 1
                    print(f"  No improvement ({no_improve}/{ES_PATIENCE})")
                    if no_improve >= ES_PATIENCE:
                        print("Early stopping.")
                        break

    avg_train = epoch_loss / max(n_steps, 1)
    print(f"\nEpoch {epoch} | Avg train: {avg_train:.4f}")
    if no_improve >= ES_PATIENCE:
        break

elapsed_total = (time.time() - start_time) / 60
print(f"\nTraining complete in {elapsed_total:.1f} min | Best val: {best_val:.4f}")

Steps per epoch : 6,900
Effective batch : 128
Total steps     : 34,500
Peak LR         : 1.0e-04
Min LR          : 1.0e-06
fp16            : True

Training NanoQA v4 | 883,166 samples | 5 epochs
Step   500 | Ep 1/5 | Train 6.7687 | Val 5.7152 | PPL 303.43 | LR 1.00e-04 | 14.2min
  ✓ Best saved (val=5.7152)
Step  1000 | Ep 1/5 | Train 5.9213 | Val 4.6309 | PPL 102.61 | LR 9.99e-05 | 28.4min
  ✓ Best saved (val=4.6309)
Step  1500 | Ep 1/5 | Train 5.3612 | Val 4.0581 | PPL 57.86 | LR 9.98e-05 | 42.6min
  ✓ Best saved (val=4.0581)
Step  2000 | Ep 1/5 | Train 4.9509 | Val 3.6862 | PPL 39.89 | LR 9.95e-05 | 56.9min
  ✓ Best saved (val=3.6862)
Step  2500 | Ep 1/5 | Train 4.6327 | Val 3.4285 | PPL 30.83 | LR 9.91e-05 | 71.1min
  ✓ Best saved (val=3.4285)
Step  3000 | Ep 1/5 | Train 4.3705 | Val 3.2575 | PPL 25.99 | LR 9.87e-05 | 85.4min
  ✓ Best saved (val=3.2575)
Step  3500 | Ep 1/5 | Train 4.1539 | Val 3.1499 | PPL 23.33 | LR 9.81e-05 | 99.6min
  ✓ Best saved (val=3.1499)
Step  4000 | Ep 1/5

In [11]:
# ── CELL 11: Save training history and curves ─────────────────────────────────
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

if len(history) >= 2:
    steps = [h["step"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("NanoQA v4 — Training Curves", fontsize=14, fontweight='bold')
    ax1.plot(steps, [h["train"] for h in history], label="Train", color="#60a5fa", lw=2)
    ax1.plot(steps, [h["val_loss"] for h in history], label="Val", color="#f87171", lw=2)
    ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
    ax1.set_title("Combined Loss (Focal + KL Distillation)")
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(steps, [h["perplexity"] for h in history], color="#4ade80", lw=2)
    ax2.set_xlabel("Step"); ax2.set_ylabel("Perplexity")
    ax2.set_title("Validation Perplexity"); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("Curves saved.")


Curves saved.


In [12]:
# ── CELL 12: Test Set Evaluation (Touched Only Once) ──────────────────────────
# Load best checkpoint and evaluate on the held-out test set.
# Test set = original pairs only, no augmentation, never seen during training.

print("Loading best checkpoint...")
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Best checkpoint: epoch {ckpt['epoch']}, step {ckpt['step']}, val_loss {ckpt['val_loss']:.4f}")

# Verify the saved prefix matches
saved_prefix = ckpt.get("answer_prefix", "NOT SAVED")
print(f"Saved ANSWER_PREFIX: {repr(saved_prefix)}")
assert saved_prefix == ANSWER_PREFIX, f"Prefix mismatch: saved={saved_prefix}, current={ANSWER_PREFIX}"
print("✓ Prefix matches — training and inference use identical tokens")

# Evaluate test loss
test_loss, test_ppl = evaluate(test_loader)
print(f"\n{'='*55}")
print("TEST SET RESULTS (honest — never seen during training)")
print(f"{'='*55}")
print(f"Combined Loss : {test_loss:.4f}")
print(f"Perplexity    : {test_ppl:.2f}")

test_results = {
    "test_loss": test_loss,
    "test_perplexity": test_ppl,
    "best_val_loss": ckpt["val_loss"],
    "best_epoch": ckpt["epoch"],
}
with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(test_results, f, indent=2)


Loading best checkpoint...
Best checkpoint: epoch 2, step 12000, val_loss 2.7542
Saved ANSWER_PREFIX: '\nAnswer:'
✓ Prefix matches — training and inference use identical tokens

TEST SET RESULTS (honest — never seen during training)
Combined Loss : 2.8024
Perplexity    : 16.48


In [13]:
# ── CELL 13: Generative Accuracy Test ─────────────────────────────────────────
# Asks the model 20 factual questions and checks if the correct keyword appears
# in the generated answer. This is the real-world performance metric.
#
# Inference prompt format: "Question: {q}\nAnswer:"
# This EXACTLY matches ANSWER_PREFIX = "\nAnswer:" used during training.
# The model then generates " William Shakespeare." etc.

def infer(q, max_new_tokens=25, temperature=0.3, top_k=10):
    """
    Generate answer for question q.
    Returns (answer_string, confidence_score, routing_label)
    """
    model.eval()
    # ── CRITICAL: prompt ends at ANSWER_PREFIX = "\nAnswer:" ────────────────
    # Must match training format exactly (same as what was trained on)
    prompt = f"Question: {q}{ANSWER_PREFIX}"
    ids    = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

    # Verify the prompt ends with token 25 (:) — same assertion as training
    last_tok = ids[0, -1].item()
    assert last_tok == 25, f"Prompt last token should be 25 (:), got {last_tok}"

    gen_ids, confs = model.generate(
        ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_k          = top_k,
        eos_token_id   = tokenizer.eos_token_id,
    )

    ans  = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    conf = sum(confs) / len(confs) if confs else 0.0

    # Routing decision (same thresholds as paper)
    if conf >= 0.60:
        route = "SLM"
    elif conf >= 0.45:
        route = "retry→SLM"
    else:
        route = "→LLM"

    return ans, conf, route


# Sanity check one question first
print("Sanity check:")
ans, conf, route = infer("Who wrote Romeo and Juliet?")
print(f"  Answer: {repr(ans)}  conf: {conf:.3f}  route: {route}")
if "shakespeare" in ans.lower() or "william" in ans.lower():
    print("  ✓ Model is generating correctly!")
elif ans.strip() == "" or all(c in ": " for c in ans):
    print("  ✗ Empty/garbage — check ANSWER_PREFIX matches training")
else:
    print(f"  ✗ Wrong answer — model may need more training or data")

# Full 20-question test
questions = [
    ("Who wrote Romeo and Juliet?",          "shakespeare"),
    ("What is the capital of France?",        "paris"),
    ("What does CPU stand for?",              "central processing"),
    ("National animal of India?",             "tiger"),
    ("Who invented the telephone?",           "bell"),
    ("Capital of Karnataka?",                 "bengaluru"),
    ("Who wrote Harry Potter?",               "rowling"),
    ("Chemical formula of water?",            "h2o"),
    ("What is Newton's second law?",          "f"),
    ("Who founded Google?",                   "page"),
    ("When did World War 2 end?",             "1945"),
    ("Who wrote the Indian Constitution?",    "ambedkar"),
    ("Largest planet in solar system?",       "jupiter"),
    ("Who painted the Mona Lisa?",            "vinci"),
    ("What nationality was Frederic Chopin?", "polish"),
    ("Who played Forrest Gump?",              "hanks"),
    ("What is the currency of Japan?",        "yen"),
    ("How many bones are in the human body?", "206"),
    ("First person to walk on the moon?",     "armstrong"),
    ("What does NLP stand for?",              "natural language"),
]

print(f"\n{'Question':<45} {'✓/✗':>4} {'Conf':>6}  Route   Answer")
print("─" * 100)

correct = 0
for q, expected in questions:
    ans, conf, route = infer(q)
    hit = expected.lower() in ans.lower()
    correct += int(hit)
    print(f"  {'✓' if hit else '✗'} {q:<43} {conf:.3f}  [{route}] {ans}")

print("─" * 100)
pct = correct / len(questions) * 100
print(f"\nGenerative accuracy: {correct}/{len(questions)} = {pct:.0f}%")
print("Queries with conf < 0.60 route to Groq/Gemma in production.")
test_results["generative_accuracy"] = pct
with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(test_results, f, indent=2)


Sanity check:
  Answer: 'William Shakespeare.'  conf: 0.898  route: SLM
  ✓ Model is generating correctly!

Question                                       ✓/✗   Conf  Route   Answer
────────────────────────────────────────────────────────────────────────────────────────────────────
  ✓ Who wrote Romeo and Juliet?                 0.898  [SLM] William Shakespeare.
  ✓ What is the capital of France?              0.768  [SLM] Paris.
  ✓ What does CPU stand for?                    0.822  [SLM] Central Processing Unit.
  ✓ National animal of India?                   0.887  [SLM] The national animal of india is the Bengal Tiger.
  ✓ Who invented the telephone?                 0.818  [SLM] Alexander Graham Bell.
  ✓ Capital of Karnataka?                       0.928  [SLM] Bengaluru.
  ✓ Who wrote Harry Potter?                     0.959  [SLM] J.K. Rowling.
  ✓ Chemical formula of water?                  0.899  [SLM] The chemical formula of water is H2O.
  ✓ What is Newton's second law?        

In [14]:
# ── CELL 14: Files summary and download instructions ──────────────────────────
import os

output_files = [
    ("nanoqa_v4_best.pt",       "Trained model weights — use this with router.py"),
    ("test_results.json",       "Test set metrics"),
    ("training_curves.png",     "Loss curves for paper"),
    ("training_history.json",   "Full training log"),
]

print("\nFiles saved to:", OUTPUT_DIR)
total_mb = 0
for fname, desc in output_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        mb = os.path.getsize(fpath) / 1e6
        total_mb += mb
        print(f"  {fname:<30} {mb:6.1f} MB  — {desc}")
    else:
        print(f"  {fname:<30}  NOT FOUND")

print(f"  {'Total':<30} {total_mb:6.1f} MB")

print("""
\n============================================================
TO DOWNLOAD:
  Kaggle folder icon (top-right) →
  /kaggle/working/nanoqa_v4/ →
  Download: nanoqa_v4_best.pt

LOCAL USAGE (router.py already has the correct inference code):
  python router.py --model nanoqa_v4_best.pt

INFERENCE FORMAT (must match training exactly):
  ANSWER_PREFIX = "\\nAnswer:"       # NO trailing space
  prompt = f"Question: {{q}}{ANSWER_PREFIX}"
  # Model generates " William Shakespeare." — space is part of BPE token
============================================================
""")



Files saved to: /kaggle/working/nanoqa_v4
  nanoqa_v4_best.pt               495.7 MB  — Trained model weights — use this with router.py
  test_results.json                 0.0 MB  — Test set metrics
  training_curves.png               0.1 MB  — Loss curves for paper
  training_history.json             0.0 MB  — Full training log
  Total                           495.8 MB


TO DOWNLOAD:
  Kaggle folder icon (top-right) →
  /kaggle/working/nanoqa_v4/ →
  Download: nanoqa_v4_best.pt

LOCAL USAGE (router.py already has the correct inference code):
  python router.py --model nanoqa_v4_best.pt

INFERENCE FORMAT (must match training exactly):
  ANSWER_PREFIX = "\nAnswer:"       # NO trailing space
  prompt = f"Question: {{q}}{ANSWER_PREFIX}"
  # Model generates " William Shakespeare." — space is part of BPE token



In [15]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/venisatellis/new-handcrafted-qa-json/new_handcrafted_qa.json
/kaggle/input/datasets/venisatellis/nanoqa-handcrafted-qa/handcrafted_qa.json
